
<font color="#CA3532"><h1 align="left">Sistema multimodal para la detección e identificación de especies de hongos mediante visión por computador y modelos de lenguaje</h1></font>
<font color="#6E6E6E"><h2 align="left">Prueba de modelo YOLO26</h2></font>

#### David Alejandro Pedroza De Jesús

# Carga de librerias

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00


In [9]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix,classification_report, roc_auc_score, roc_curve,ConfusionMatrixDisplay
import seaborn as sns
import torch
from ultralytics import YOLO
import cv2
import shutil
from google.colab import drive

In [10]:

drive.mount('/content/drive')

Mounted at /content/drive


#   Carga de modelo YOLO26 y algunos datos importantes

In [12]:
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
shutil.copy(
    "/content/drive/MyDrive/kaggle.zip",

    "/content/")



In [ ]:
shutil.copytree(
    "/content/drive/MyDrive/runstf_8",
    "/content/runstf_8")

In [ ]:
!unzip kaggle.zip

In [ ]:
rutas_test = pd.read_csv("kaggle/working/test.csv")
info_especies = pd.read_csv("InfoEspecies.csv")
#info_especies = info_especies.drop(info_especies.columns[0], axis= "columns")
resultados = pd.read_csv(r"runsft_8\classify\train\results.csv")
model = YOLO(r"runsft_8\classify\train\weights\best.pt")
test = pd.merge(info_especies, rutas_test, on='label', how='inner')


Ahora voy a obtener predicciones del conjunto de test

In [ ]:
def get_prediction(results,labels):
    results_ls = results[0]

    probs = results_ls.probs.data.cpu().numpy()

    prob_img = probs
    pred_img = prob_img.argmax()

    return pred_img, prob_img

In [ ]:

labels = info_especies.label.unique()
y_true = []
y_pred = []
y_proba = []
for registro,real_name in zip(test.image_path,test.label):
    img = cv2.imread(registro[1:])
    results = model(img)
    results_ls = results[0]
    y_true.append(int(np.where(labels == real_name)[0][0]))
    pred, prob = get_prediction(results,labels)
    y_pred.append(pred)
    y_proba.append(prob)


#   Graficos importantes para determinar el rendimientos
##  Loss

Vemos que la función de perdida del modelo decrece conforme las epocas van pasando, más o menos parece que con 12 epocas de entrenamiento son suficientes, ya que usando más el modelo no mejora.

In [ ]:
plt.plot(resultados.epoch,resultados["train/loss"], label = "Entrenamiento")
plt.plot(resultados.epoch,resultados["val/loss"], label = "validación")
plt.legend(loc="upper right")
plt.title("Función de perdida")
plt.show()

##  Matriz de confución
### Versión normal
Vemos que además el modelo no parece requerir un ajuste de umbral debido a que todas las clases de predicen con un cierto exito

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize= "true").round(2)
plt.subplots(figsize=(60,50))
sns.heatmap(cm,xticklabels= labels, yticklabels= labels,annot=True)
plt.title("Matriz de confución")
plt.show()

### Versión veneno

In [ ]:
y_3_true = test.Info
y_3_pre = pd.merge(info_especies, pd.DataFrame(labels[y_pred],columns= ["label"]), on='label', how='inner')["Info"]

In [ ]:
cm_3 = confusion_matrix(y_3_true, y_3_pre, normalize= "true")
cm_display = ConfusionMatrixDisplay(confusion_matrix = cm_3,display_labels= y_3_true.unique())
cm_display.plot()
plt.show()

##  Curva ROC

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
y_proba = np.array(y_proba)
n_classes = y_proba.shape[1]
y_true_bin = label_binarize(y_true, classes=range(n_classes))

plt.figure(figsize=(8,6))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Clase {i} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curvas ROC')
#plt.legend()
plt.grid(True)
plt.show()

##  Metricas básicas
### Para especies

In [ ]:
print(classification_report(y_true, y_pred))

### Para comestibilidad

In [ ]:
print(classification_report(y_3_true, y_3_pre))